# Group Analytikos (Nr. 22): Data Analysis Project - Crime in Chicago from 2012 to 2017
Group Members: 
1. Yen Vu Thi Ngoc
2. Yesenia
3. Arpita
4. Chan Surendra

### Table of Contents
1. Introduction
2. Load dataset and import neccessary libraries/modules
3. Data cleaning, preparation and explorative data analysis 
4. Data pre-processing and model development
5. Evaluation techniques and metrics
6. Tuning metrics
7. Conclusions

### 1. Introduction
Project Goals and Problem Statement

### 2. Load Dataset and import neccessary libraries / modules (Yese, Arpita)

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime


In [ ]:
#The csv file should be in the same direction as the python file
chicago_crimes = pd.read_csv("Chicago_Crimes_2012_to_2017.csv")
chicago_crimes

In [ ]:
chicago_crimes.info()

In [ ]:
chicago_crimes.shape

### 3. Data cleaning, preparation and explorative data analysis 

#### 3.1 Data cleaning and preparation (Yese, Arpita)

In [ ]:
missing_values = chicago_crimes.isnull().sum()
print(missing_values)

Mising values:
1. Case number (1): Is an identifier of the case. A random number and a letter can work to replace the missing value.
2. Ward (14): is a administrative division, each ward has its own municipality. The ward could be associate with the district. The missing wards belong to the same district (16). The most commun ward for Beat 1654 and 1614 is 41.
3. Community area (40): Is related to the beat. The propose way to eliminate the missing values is to find the most commun value of community are according to the beat. (The same beat can have several values of community area, but has a predominant one). Beat is a small zone of the city like blocks. Coummity area represent a bigger region.
4. Location description (1658): The missing values could be replace with OTHER, which is a commun and general location description.
5. District (1): with the filters apply, the most appropiate district would be 3.
6. Coordinate (5 columns with 37083 missing values each): Get the information base on the column Block with library in python. Approach 1: using geopy library, but takes a lot of time to obtain the information. Approach 2: Make a dictionary of the actual information of the coordantes of the actual data, by Block.

In [ ]:
#Rows with missing values
row_with_nulls = chicago_crimes[chicago_crimes.isnull().any(axis=1)]
#print(row_with_nulls)
#row_with_nulls.to_csv(r"C:\Users\yesem\Dropbox\1 University\2 Semester\Python and Advanced Data Science\Final Project\filas_con_nulos.csv", index=False)

#2. Rows with missing values in column Ward
row_with_nulls_ward = chicago_crimes[chicago_crimes['Ward'].isnull()]
#print(row_with_nulls_ward[['Ward', 'Beat','District']])

#5. Rows with missing values in column District
row_with_nulls_district = chicago_crimes[chicago_crimes['District'].isnull()]
#print(row_with_nulls_district[['Ward', 'Beat','District']])

#5. The missing value in district have ward=7 and Beat=334, filter this parameters to check the possible district values
value_missing_district = chicago_crimes[
    (chicago_crimes['Ward'] == 7) & 
    (chicago_crimes['Beat'] == 334)
]
print(value_missing_district[['Ward', 'Beat','District']])
count_values_distinc = value_missing_district[['Ward', 'Beat','District']].value_counts()
print(count_values_distinc)
#The most commun district is 3 for ward 7 and Beat 334.

#value_missing_district_unic = value_missing_district[['Ward', 'Beat','District']].drop_duplicates()
#print(value_missing_district_unic)

#3. Comunnity area

beat_to_common_community = {}

#Group by beat
grouped = chicago_crimes.groupby('Beat')

# Community Area most frequent
for beat, group in grouped:
    common_community_area = group['Community Area'].mode().iloc[0] 
    beat_to_common_community[beat] = common_community_area

print("Dictionary of Beat and Community Area most commun:")
print(beat_to_common_community)



In [ ]:
df=chicago_crimes

In [ ]:
#6. Approach 1
#pip install geopy
#!pip install pyproj

In [ ]:
#6. Approach 1
#from geopy.geocoders import Nominatim
#from pyproj import Proj, transform
#import pandas as pd
#import time

# Inicializar el geocodificador
#geolocator = Nominatim(user_agent="chicago_crimes_geocoder")

# Inicializar la proyección para convertir a X e Y
#wgs84 = Proj(proj="latlong", datum="WGS84")  # Sistema geográfico WGS84
#state_plane = Proj(init="epsg:3435")  # Sistema Estatal de Illinois Este

# Función para obtener las coordenadas de un bloque
#def get_coordinates_from_block(block):
#    try:
#        location = geolocator.geocode(f"{block}, Chicago, IL")
#        if location:
 #           return location.latitude, location.longitude
 #       else:
  #          return None, None
   # except Exception as e:
    #    print(f"Error al geocodificar {block}: {e}")
     #   return None, None
# Crear un diccionario para almacenar coordenadas por bloque
#block_coordinates = {}

# Iterar sobre los bloques únicos
#unique_blocks = df['Block'].unique()
#for block in unique_blocks:
#    if block not in block_coordinates:  # Solo buscar si no está en el diccionario
#        lat, lon = get_coordinates_from_block(block)
#        block_coordinates[block] = (lat, lon)

#print("Diccionario de coordenadas por bloque:")
#print(block_coordinates)

In [ ]:
#6. Approach 2:
#import pandas as pd
from collections import defaultdict, Counter

# Create a dictionary with default value being a dictionary of empty lists
block_data = defaultdict(lambda: {'X Coordinate': [], 'Y Coordinate': [], 'Latitude': [], 'Longitude': [], 'Location': []})

# Iterate over each row in the DataFrame
for _, row in df.iterrows():
    block = row['Block']
    x = row['X Coordinate']
    y = row['Y Coordinate']
    latitude = row['Latitude']
    longitude = row['Longitude']
    location = row['Location']
    
    # Check if the coordinates are not NaN
    if pd.notna(x) and pd.notna(y) and pd.notna(latitude) and pd.notna(longitude) and pd.notna(location):
        # Add the coordinates to the respective lists in block_data
        block_data[block]['X Coordinate'].append(x)
        block_data[block]['Y Coordinate'].append(y)
        block_data[block]['Latitude'].append(latitude)
        block_data[block]['Longitude'].append(longitude)
        block_data[block]['Location'].append(location)

# Count the frequency of each combination in each Block
block_summary = {block: {"details": details, "counts": len(details['X Coordinate'])} for block, details in block_data.items()}

# Print the summary of details for each block
for block, info in block_summary.items():
    print(f"Block: {block}")
    print(f"Details: {info['details']}")
    print(f"Unique combination count: {info['counts']}")
    print("-" * 40)

# Create a final dictionary with the most common coordinates for each Block
block_most_common = {}

for block, details in block_data.items():
    # Combine coordinates and location information into tuples
    coords = list(zip(details['X Coordinate'], details['Y Coordinate'], details['Latitude'], details['Longitude'], details['Location']))
    # Count the frequency of each coordinate combination
    freq = Counter(coords)
    
    # Select the most common combination
    most_common_coords = freq.most_common(1)[0][0]  # The first element of the tuple is the coordinates
    block_most_common[block] = most_common_coords

# Print the dictionary of most common coordinates for each Block
print("Dictionary with the most common coordinates by Block:")
for block, coords in block_most_common.items():
    print(f"Block: {block}, Most common coordinates: {coords}")

In [ ]:
#Make changes to the data set to deal with missing values.
df_copy = df.copy()
#1. Case number
df_copy['Case Number'].fillna("JE999999", inplace=True)
#5. District
df_copy['District'].fillna("3", inplace=True)
#2.Wards
df_copy['Ward'].fillna("41", inplace=True)
#3. Community area
df_copy['Community Area'] = df.apply(
    lambda row: beat_to_common_community[row['Beat']] if pd.isna(row['Community Area']) else row['Community Area'],
    axis=1
)
#4. Location description
df_copy['Location Description'].fillna("OTHER", inplace=True)
#6 Coordinates

# Function to fill missing values based on block_most_common dictionary
def fill_missing_values(row):
    block = row['Block']
    # If the block has a most common coordinate combination, fill the missing values
    if block in block_most_common:
        most_common_coords = block_most_common[block]
        # Iterate over the columns to fill missing values
        for col, value in zip(['X Coordinate', 'Y Coordinate', 'Latitude', 'Longitude', 'Location'], most_common_coords):
            if pd.isna(row[col]):  # Only fill if the value is NaN
                row[col] = value
    return row

# Apply the function to fill missing values
df_copy = df_copy.apply(fill_missing_values, axis=1)

missing_values_2 = df_copy.isnull().sum()
print(missing_values_2)

In [ ]:
row_with_nulls_2 = df_copy[df_copy.isnull().any(axis=1)]
print(row_with_nulls_2)
#row_with_nulls_2.to_csv(r"C:\Users\yesem\Dropbox\1 University\2 Semester\Python and Advanced Data Science\Final Project\filas_con_nulos_2.csv", index=False)

6. Still some missing values in the coordinates. After applying the most comun cordinates for each Block, the next approach is with the most comun coordinate for each Beat

In [ ]:
# Create a dictionary with default value being a dictionary of empty lists
Beat_data = defaultdict(lambda: {'X Coordinate': [], 'Y Coordinate': [], 'Latitude': [], 'Longitude': [], 'Location': []})

# Iterate over each row in the DataFrame
for _, row in df_copy.iterrows():
    beat = row['Beat']
    x = row['X Coordinate']
    y = row['Y Coordinate']
    latitude = row['Latitude']
    longitude = row['Longitude']
    location = row['Location']
    
    # Check if the coordinates are not NaN
    if pd.notna(x) and pd.notna(y) and pd.notna(latitude) and pd.notna(longitude) and pd.notna(location):
        # Add the coordinates to the respective lists in block_data
        Beat_data[beat]['X Coordinate'].append(x)
        Beat_data[beat]['Y Coordinate'].append(y)
        Beat_data[beat]['Latitude'].append(latitude)
        Beat_data[beat]['Longitude'].append(longitude)
        Beat_data[beat]['Location'].append(location)

# Count the frequency of each combination in each Block
beat_summary = {beat: {"details": details, "counts": len(details['X Coordinate'])} for beat, details in Beat_data.items()}

# Print the summary of details for each block
for beat, info in beat_summary.items():
    print(f"Beat: {beat}")
    print(f"Details: {info['details']}")
    print(f"Unique combination count: {info['counts']}")
    print("-" * 40)

# Create a final dictionary with the most common coordinates for each Block
beat_most_common = {}

for beat, details in Beat_data.items():
    # Combine coordinates and location information into tuples
    coords = list(zip(details['X Coordinate'], details['Y Coordinate'], details['Latitude'], details['Longitude'], details['Location']))
    # Count the frequency of each coordinate combination
    freq = Counter(coords)
    
    # Select the most common combination
    most_common_coords_beat = freq.most_common(1)[0][0]  # The first element of the tuple is the coordinates
    beat_most_common[beat] = most_common_coords_beat

# Print the dictionary of most common coordinates for each Block
print("Dictionary with the most common coordinates by Beat:")
for beat, coords in beat_most_common.items():
    print(f"Beat: {beat}, Most common coordinates: {coords}")

In [ ]:
# Function to fill missing values based on beat_most_common dictionary
def fill_missing_values_2(row):
    beat = row['Beat']
    # If the block has a most common coordinate combination, fill the missing values
    if beat in beat_most_common:
        most_common_coords_beat = beat_most_common[beat]
        # Iterate over the columns to fill missing values
        for col, value in zip(['X Coordinate', 'Y Coordinate', 'Latitude', 'Longitude', 'Location'], most_common_coords_beat):
            if pd.isna(row[col]):  # Only fill if the value is NaN
                row[col] = value
    return row

# Apply the function to fill missing values
df_copy = df_copy.apply(fill_missing_values_2, axis=1)

missing_values_3 = df_copy.isnull().sum()
print(missing_values_3)

In [ ]:
print(df_copy.dtypes)
df_copy.info()

#### 3.2 Explorative data analysis (Responsible member: Yen Vu Thi Ngoc)

In [ ]:
crimes_clean = df_copy.copy()
crimes_clean

In [ ]:
crimes_clean.info()

In [ ]:
duplicate_case_numbers = crimes_clean['Case Number'].value_counts()
duplicate_case_numbers = duplicate_case_numbers[duplicate_case_numbers > 1]
print(duplicate_case_numbers)

In [ ]:
#1. Create columns for Year, Month, Date, Hour, Minute, Second
crimes_clean['Date'] = pd.to_datetime(crimes_clean['Date'])
crimes_clean['Year'] = crimes_clean['Date'].dt.year
crimes_clean['Month'] = crimes_clean['Date'].dt.month
crimes_clean['Day'] = crimes_clean['Date'].dt.day
crimes_clean['Hour'] = crimes_clean['Date'].dt.hour
crimes_clean['Minute'] = crimes_clean['Date'].dt.minute
crimes_clean['Second'] = crimes_clean['Date'].dt.second

In [ ]:
# Descriptive statistics
descriptive_stats = crimes_clean.describe(include='all')
print(descriptive_stats)

In [ ]:
# Distribution of primary types of crimes
plt.figure(figsize=(8, 6))
sns.countplot(y='Primary Type', data=crimes_clean, order=crimes_clean['Primary Type'].value_counts().index, palette='viridis')
plt.title('Distribution of Primary Types of Crimes')
plt.xlabel('Count')
plt.ylabel('Primary Type')
plt.show()

# Group the data by 'Year' and 'Primary Type' and count the occurrences
crime_counts = crimes_clean.groupby(['Year', 'Primary Type']).size().unstack()

# Sort the columns based on the total number of crimes for each type
crime_counts = crime_counts[crime_counts.sum().sort_values(ascending=False).index]

# Plot the line graph with sorted legend for top 10 crimes
crime_counts_top_10 = crime_counts[crime_counts.columns[:10]]
plt.figure(figsize=(8, 6))
crime_counts_top_10.plot(kind='line', figsize=(8, 6), colormap='tab20', linewidth=2)
plt.title('Number of Crimes by Top 10 Types Over the Years')
plt.xlabel('Year')
plt.ylabel('Number of Crimes')
plt.legend(title='Primary Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.show()

In [ ]:
# Distribution of locations where crimes occur most frequently (top 5)
plt.figure(figsize=(8, 6))
sns.countplot(y='Location Description', data=crimes_clean, order=crimes_clean['Location Description'].value_counts().index[:10], palette='icefire')
plt.title('Distribution of Locations where Crimes Occur Most Frequently')
plt.xlabel('Count')
plt.ylabel('Location Description')
plt.show()


# Pie chart of arrests with percentage, color like bright viridis
arrests = crimes_clean['Arrest'].value_counts()
plt.figure(figsize=(8, 6))
plt.pie(arrests, labels=arrests.index, autopct='%1.1f%%', colors=sns.color_palette('icefire'), startangle=90)
plt.title('Arrests Distribution')
plt.axis('equal')
plt.show()

# Pie chart of domestic with percentage, color like bright viridis
domestic = crimes_clean['Domestic'].value_counts()
plt.figure(figsize=(8, 6))
plt.pie(domestic, labels=domestic.index, autopct='%1.1f%%', colors=sns.color_palette('icefire'), startangle=90)
plt.title('Domestic Distribution')
plt.axis('equal')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a figure with a 2x2 grid for subplots
plt.figure(figsize=(12, 8))

# Subplot 1: Distribution of crimes over the years
plt.subplot(2, 2, 1)  # 2 rows, 2 columns, first subplot
sns.countplot(x='Year', data=crimes_clean, palette='crest')
plt.title('Distribution of Crimes Over the Years')
plt.xlabel('Year')
plt.ylabel('Count')

# Subplot 2: Distribution of crimes by month
plt.subplot(2, 2, 2)  # 2 rows, 2 columns, second subplot
sns.countplot(x='Month', data=crimes_clean, palette='crest')
plt.title('Distribution of Crimes by Month')
plt.xlabel('Month')
plt.ylabel('Count')

# Subplot 3: Distribution of crimes by day of the week
crimes_clean['DayOfWeek'] = crimes_clean['Date'].dt.day_name()  # Ensure 'DayOfWeek' is calculated before plotting
plt.subplot(2, 2, 3)  # 2 rows, 2 columns, third subplot
sns.countplot(x='DayOfWeek', data=crimes_clean, order=['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'], palette='crest')
plt.title('Distribution of Crimes by Day of the Week')
plt.xlabel('Day of the Week')
plt.ylabel('Count')

# Subplot 4: Distribution of crimes by hour
plt.subplot(2, 2, 4)  # 2 rows, 2 columns, fourth subplot
sns.countplot(x='Hour', data=crimes_clean, palette='crest')
plt.title('Distribution of Crimes by Hour')
plt.xlabel('Hour')
plt.ylabel('Count')

# Adjust layout to avoid overlap and improve the appearance
plt.tight_layout()

# Show the combined plot
plt.show()

In [ ]:
crimes_clean.columns

In [ ]:
# Select the specified columns
columns_of_interest = ['ID', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'Year']
# Calculate the correlation matrix
correlation_matrix = crimes_clean[columns_of_interest].corr()
# Create a heatmap of the correlation matrix
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap="icefire", fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix')
plt.show()

In [ ]:
#Using Folium to create interactive heatmap for top crime hotspots 
import folium
from folium.plugins import HeatMap

# Filter out rows with missing Latitude or Longitude
new_df = crimes_clean.dropna(subset=['Latitude', 'Longitude'])

# Create a list of lists containing the latitude and longitude of each crime
heat_data = [[row['Latitude'], row['Longitude']] for index, row in new_df.iterrows()]

# Create a base map centered around the city of Chicago (using an approximate central point)
map = folium.Map(location=[41.8781, -87.6298], zoom_start=12)

# Add HeatMap to the map
HeatMap(heat_data).add_to(m)

# Save the map to an HTML file (or display it if using Jupyter Notebook)
#map.save('crime_heatmap.html')
map

#### Summary about Types of Crimes, Locations, Number of Arrests, Domestic Crimes or Time when Crimes occur
- **Types of Crimes**: Theft, Battery and Criminal Damage are common crimes.Narcotics, Assault, and drug-related Offenses are also relatively frequent, while Crimes like Homicide, Arson, and Kidnapping are relatively rare.
- **Locations**: Streets, residences and apartments are the most common location for crimes probably because streets are public spaces where people and vehicles pass through frequently. Sidewalks, parking lots, and alleys are also relatively common while other locations like small retail stores and schools are less common. 
- **Number of Arrests**: A significantly larger number of crimes did not result in an arrest (74.1%). A smaller but still substantial number of crimes resulted in an arrest. 
- **Number of Domestic Crimes**: A significantly larger number of crimes were not domestic in nature (84.9%), meaning do not involve domestic incidents. 
- **Crime Trends over time**: 
    - The graph shows a general decline in the number of crimes from 2012 to 2017. However, it's important that the data for 2017 is incomplete, which might skew the overall trend.
    - Seasonal Variations: The graph illustrates a clear seasonal pattern in crime occurrences. The months of July and August tend to have the highest number of reported crimes, while February and March have the lowest. This seasonal variation could be attributed to various factors, such as increased outdoor activities and longer daylight hours during summer months.
    - Weekly Patterns: It is shown that Friday and Saturday have the highest number of reported crimes, while Sunday has the lowest. This pattern might be influenced by increased social activity and alcohol consumption during weekends.
    - Hourly Trends: The peak hours for crime occurrences are between 8 PM and 1 AM. This suggests that a significant portion of criminal activity happens during the evening and early morning hours.
- **Correlation between Variables**: The majority of the correlation coefficients are close to zero, indicating weak or no linear relationships between the variables. This suggests that these variables are largely independent of each other.

### 4. Data pre-processing and model development

### 5. Evaluation techniques and metrics

### 6. Tuning metrics

### 7. Conclusions